# 07 · Kubernetes, GPU sharing (HAMi) and autoscaling (KEDA)

Phases 0 to 6 ran on Docker Compose. This phase moves the stack onto **Kubernetes (k3s)** and asks two operational questions:

1. **Can two models share one GPU?** HAMi slices a GPU between pods with memory (and optionally compute) limits. What does sharing cost the busy model, and does it hurt the quiet one?
2. **Can the cluster add GPUs when load rises?** KEDA scales the vLLM Deployment from 1 to 2 pods on the number of requests inside vLLM, and back down when load falls.

All numbers are read from `results/07_gpu_slicing_hami/`.

## Setup

* **Cluster:** one node, k3s with NVIDIA as the default container runtime; HAMi for GPU sharing; KEDA for autoscaling (`scripts/setup_k3s.sh`).
* **Stack as Kubernetes resources** (`scripts/k8s_manifests.py`): gateway (8 workers), NGINX (least connections to vLLM pods found through a headless Service), Prometheus (finds vLLM pods through the Kubernetes API), Grafana, Jaeger, GPU exporter.
* **Tenant A**: Qwen2.5-7B-Instruct, FP8 weights (the Phase 5 config), under Locust load through the gateway: 32 → 256 users.
* **Tenant B**: Qwen2.5-1.5B-Instruct, a latency-sensitive internal service: 4 clients sending short questions the whole time, starting 60 s before A's load.

| Scenario | GPUs | Tenant A | Tenant B |
|---|---|---|---|
| isolated | 2 | whole GPU 0 | whole GPU 1 |
| shared | 1 | GPU 0, 20,000 MB limit | GPU 0, 9,000 MB limit |
| shared_cores | 1 | GPU 0, 20,000 MB, 70% of cores | GPU 0, 9,000 MB, 30% of cores |
| autoscale | 1 → 2 | 1 to 2 pods, whole GPUs, KEDA | none |

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = ROOT / "results" / "07_gpu_slicing_hami"
sys.path.insert(0, str(ROOT / "loadtest"))
from results_logger import load_runs

runs = load_runs("07_gpu_slicing_hami")
sweeps = {r["config"]["label"]: r for r in runs if r["name"].startswith("load_sweep_")}
tenants = {r["config"]["label"]: r for r in runs if r["name"].startswith("tenants_")}
timeline = next((r for r in reversed(runs) if r["name"] == "timeline_autoscale"), None)
print(f"sweeps: {sorted(sweeps)} | tenant analyses: {sorted(tenants)} | autoscale timeline: {bool(timeline)}")
for f in sorted(RESULTS.glob("*/failed.log")):
    print(f"DID NOT START: {f.parent.name}")

def v(d, *keys):
    for k in keys:
        d = (d or {}).get(k)
    return d

def cell(value, fmt, width, scale=1, unit=""):
    return ("-" if value is None else format(value * scale, fmt) + unit).rjust(width)

## Where HAMi put the pods

In [ ]:
for s in ("isolated", "shared", "shared_cores"):
    f = RESULTS / s / "placement.txt"
    if f.exists():
        lines = f.read_text().splitlines()
        print(f"--- {s}")
        print("\n".join(l for l in lines if "vllm-" in l or "MiB /" in l or "python" in l.lower())[:1500])

## Tenant A: the busy model

In [ ]:
scen = [s for s in ("isolated", "shared", "shared_cores") if s in sweeps]
if scen:
    users = [s["users"] for s in sweeps[scen[0]]["metrics"]["steps"]]
    print(f"{'users':>6}" + "".join(f"{s + ' tok/s':>20}{'TTFT p95':>10}{'ITL p50':>9}" for s in scen))
    by = {s: {x["users"]: x for x in sweeps[s]["metrics"]["steps"]} for s in scen}
    for u in users:
        line = f"{u:>6}"
        for s in scen:
            x = by[s].get(u, {})
            line += f"{cell(v(x, 'server', 'output_tokens_per_s'), '.0f', 20)}{cell(v(x, 'ttft_s', 'p95'), '.0f', 10, 1000, 'ms')}{cell(v(x, 'itl_s', 'p50'), '.1f', 9, 1000, 'ms')}"
        print(line)
else:
    print("No runs yet: run scripts/run_phase7.sh on the GPU box, then copy results/ back.")

## Tenant B: the quiet neighbour

In [ ]:
if tenants:
    for s in [x for x in ("isolated", "shared", "shared_cores") if x in tenants]:
        print(f"--- {s}")
        print(f"{'window':<18}{'B TTFT p50':>11}{'B TTFT p95':>11}{'B e2e p95':>10}{'B errors':>9}")
        for w in tenants[s]["metrics"]["windows"]:
            b = w["b"]
            print(f"{w['window']:<18}{cell(b['ttft_s'].get('p50'), '.0f', 11, 1000, 'ms')}{cell(b['ttft_s'].get('p95'), '.0f', 11, 1000, 'ms')}"
                  f"{cell(b['e2e_s'].get('p95'), '.0f', 10, 1000, 'ms')}{b['errors']:>9}")
    fig, ax = plt.subplots(figsize=(10, 4))
    for s in [x for x in ("isolated", "shared", "shared_cores") if x in tenants]:
        ws = tenants[s]["metrics"]["windows"]
        ax.plot([w["a_users"] for w in ws], [w["b"]["ttft_s"].get("p95") for w in ws], "o-", label=s)
    ax.set_xlabel("tenant A users (0 = B alone)"); ax.set_ylabel("tenant B TTFT p95 (s)")
    ax.set_title("Does the busy neighbour slow the quiet one?"); ax.legend()
    plt.tight_layout(); plt.show()

## Cost of sharing

Isolated uses 2 GPUs, shared uses 1. Output tokens per GPU at the highest step, both tenants together.

In [ ]:
for s in [x for x in ("isolated", "shared", "shared_cores") if x in sweeps]:
    last = sweeps[s]["metrics"]["steps"][-1]
    gpus = 2 if s == "isolated" else 1
    a = v(last, "server", "output_tokens_per_s")
    print(f"{s:<14}{gpus} GPU(s)   tenant A {a:>7.0f} tok/s at {last['users']} users   per GPU {a / gpus:>7.0f}")

## Autoscaling with KEDA

In [ ]:
rep_file = RESULTS / "autoscale" / "replicas.jsonl"
if timeline and rep_file.exists():
    reps = [json.loads(l) for l in rep_file.read_text().splitlines() if l.strip()]
    start = timeline["load"]["start"]
    tl = timeline["metrics"]["timeline"]
    steps, step_s = timeline["load"]["steps"], timeline["load"]["step_seconds"]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.step([r["t"] - start for r in reps], [r["replicas"] for r in reps], where="post", label="pods (desired)")
    ax.step([r["t"] - start for r in reps], [r["ready"] or 0 for r in reps], where="post", label="pods ready")
    ax.set_ylim(0, 3); ax.set_ylabel("vLLM pods"); ax.set_xlabel("seconds")
    ax2 = ax.twinx()
    ax2.plot([b["t_s"] for b in tl], [b["ttft_p95_s"] for b in tl], color="#d94801", label="TTFT p95")
    ax2.set_ylabel("TTFT p95 (s)")
    ax3 = ax.twiny(); ax3.set_xlim(ax.get_xlim())
    ax3.set_xticks([i * step_s for i in range(len(steps))]); ax3.set_xticklabels(steps, fontsize=7); ax3.set_xlabel("Locust users")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.show()
    print(f"{timeline['metrics']['requests']} requests, {timeline['metrics']['errors']} errors")
    ev = RESULTS / "autoscale" / "scale_events.txt"
    if ev.exists():
        print(ev.read_text()[:1500])
else:
    print("No autoscale run.")

## Grafana

*(added after the GPU session)*

## What happened

*(written after the GPU session, from the numbers above)*

## Phase 7 exit criteria

- [x] k3s with NVIDIA runtime, HAMi and KEDA installed by one script
- [x] The stack as Kubernetes resources, with Prometheus finding vLLM pods itself
- [x] Isolated, shared and shared with core limits scenarios, with a latency probe for the quiet tenant
- [x] KEDA autoscaling on requests in vLLM
- [ ] Phase 7 session on a 2x RTX 5090 machine
- [ ] Cost of sharing, noisy neighbour effect and autoscaling behaviour explained